In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import numpy as np
import pandas as pd
import librosa as lr
import librosa.display as lrd
import matplotlib.pyplot as plt
import IPython.display as ipd

from pathlib import Path

import tdseries as td
from plots.timeframe import plot_timeframe
from plots.timeframe.renderers import make_spectrogram_series

from models.rps_predictor import SimpleConvV2

In [ ]:
def _load_michaels_data_raw(wav_path, csv_path, time_offset = 0.0, time_dilation = 1.0, sr=None):
    wav, sample_rate = lr.load(wav_path, sr=sr, mono=False)
    if len(wav.shape) == 1:
        wav = wav[None, :]
    
    csv = pd.read_csv(csv_path)
    ms_cols = [c for c in csv.columns if 'Motor' in c][:4]
    t_col = 'Clock:offsetTime'
    small_csv = csv[[t_col] + ms_cols]
    
    wav_duration = wav.shape[-1] / sample_rate
    cut_csv = small_csv[(small_csv[t_col] >= time_offset) & (small_csv[t_col] <= wav_duration + time_offset)]
    ts = cut_csv[t_col].values
    
    if ts[0] > time_offset:
        print(ts[0] - time_offset)
        wav = wav[:, int((ts[0] - time_offset) * sample_rate):]
        time_offset = ts[0]
        wav_duration = wav.shape[-1] / sample_rate        
    
    if ts[-1] < wav_duration + time_offset:
        wav = wav[:, :int((ts[-1] - ts[0])*sample_rate)]

    jump_idx = np.argmax(np.diff(ts))
    print(jump_idx)
    # ts = np.linspace(ts[0], ts[-1], len(ts))
    ts[0:jump_idx+2] = np.linspace(ts[0], ts[jump_idx+2], jump_idx+2)
    ts *= time_dilation
    
    ms = cut_csv[ms_cols].values.T / 60
    return wav, ts, ms, sample_rate

In [ ]:
def display_michaels_data(
    wav_path,
    csv_path,
    time_offset=0.0,
    time_dilation=1.0,
    sr=None,
    display_offset=0.0,
    display_duration=10.0,
    channel=0,
):
    wav, ts, ms, sr = _load_michaels_data_raw(
        wav_path,
        csv_path,
        time_offset=time_offset,
        time_dilation=time_dilation,
        sr=sr,
    )
    print(wav.shape)
    print(ts.shape)
    print(ms.shape)
    print(sr)

    audio = td.uniform(wav[channel], sr, dims=("time",), t_start=0.0)
    rps = td.events(ts, ms, dims=("rotor", "time"))
    tf = td.Frame({
        "audio": audio,
        "rps": rps,
    })

    print(ts)
    slc = tf.time[display_offset:display_offset + display_duration]
    # Spectrograms are plot-only PlotTracks (not data-model Frame entries), so
    # compute it from the sliced audio and pass it alongside the entry names.
    spec_track = make_spectrogram_series(slc["audio"], fmax=None)
    _ = plot_timeframe(slc, tracks=["audio", spec_track, "rps"])

In [ ]:
display_michaels_data(
    '../data/recording_with_motor_speed/recording_1/124.wav',
    '../data/recording_with_motor_speed/recording_1/FLY124.csv',
    time_offset=-20.84,
    time_dilation=1.001,
    sr=16000,
    display_offset=0.0,
    display_duration=10.0,
)

In [ ]:
display_michaels_data(
    '../data/recording_with_motor_speed/recording_2/125.wav',
    '../data/recording_with_motor_speed/recording_2/FLY125.csv',
    time_offset=-26.51,
    time_dilation=1.0048,
    sr=16000,
    display_offset=15.0,
    display_duration=156.0
)

In [ ]:
display_michaels_data(
    '../data/new-drone-noises/103_2.wav',
    '../data/new-drone-noises/FLY103.csv',
    time_offset=0.0,
    time_dilation=1.00,
    sr=16000,
    display_offset=0.0,
    display_duration=10.0
)

In [ ]:
D = np.abs(lr.stft(wav[0]))
D_db = lr.amplitude_to_db(D, ref=np.max)

plt.figure(figsize=(10, 4))
lrd.specshow(D_db, sr=sr, x_axis='time', y_axis='linear', cmap='coolwarm')
plt.colorbar(format='%+2.0f dB')
plt.show()

In [ ]:
ipd.display(ipd.Audio(data=wav[0], rate=sr))

In [ ]:
df = pd.read_csv('../data/recording_with_motor_speed/recording_1/FLY124.csv')
df

In [ ]:
list(df.columns)

In [ ]:
ms_cols = [col for col in df.columns if col.startswith('Motor:Speed')]
ms_cols

In [ ]:
t_col = 'Clock:offsetTime'
small_df = df[[t_col] + ms_cols]
small_df

In [ ]:
MICHAELS_FILES = [
    # ("103_2.wav", "FLY103.csv", -0.94, 12.0, 100.0),
    # ("108_2.wav", "FLY108.csv", -0.40, 9.0, 88.0),
    ("124.wav", "FLY124.csv", -20.63, 33.0, 105.0),
    ("125.wav", "FLY125.csv", -26.27, 17.0, 170.0),
]

In [ ]:
tick_col = 'Clock:Tick#'

In [ ]:
ticks = df[tick_col][1:]
times = df[t_col][1:]

In [ ]:
np.diff(times)

In [ ]:
plt.plot(np.arange(199), np.diff(times[3100:3300]))

In [ ]:
times[3200:3300]

In [ ]:
plt.plot(np.arange(len(ticks)-1), np.diff(ticks))